# MXFP8 Block Operations with Allo

This tutorial shows how to use the MXFP8 library kernels in Allo:

1. Encode float32 data into MXFP8 blocks (Python reference model)
2. Run `mxfp8_block_add` on the CPU via LLVM simulation
3. Generate Vivado HLS C++ from the same kernel
4. Apply a simple schedule to pipeline the loops

**Prerequisites**: `conda activate allo` and a built Allo install (`pip install -v -e .`).

## MXFP8 block layout

Each MXFP8 block contains 32 elements:

- **Scale**: one `E8M0` byte (shared block exponent, power-of-two only)
- **Data**: 32 × `E4M3` bytes (4-bit exponent + 3-bit mantissa per element)

In Allo we store this as:

```python
scale: uint8          # E8M0
data:  uint8[32]      # E4M3 × 32
```

The library provides both a **Python reference** (`ref_*` functions) for golden-model
verification and **Allo kernels** (`mxfp8_block_add`, etc.) that compile to LLVM / HLS.

In [8]:
import numpy as np
import allo
from pathlib import Path
from IPython.display import Code, display
from allo.library import mxfp8
from allo.library.mxfp8 import (
    MXFP8_BLOCK_SIZE,
    ref_encode_block,
    ref_decode_block,
)

## Step 1 — Prepare MXFP8 inputs with the reference encoder

We encode two random float32 vectors into MXFP8 blocks using the NumPy reference
implementation. This gives us test inputs and a golden `a + b` result.

In [9]:
bs = MXFP8_BLOCK_SIZE  # 32
np.random.seed(0)

a = np.random.randn(bs).astype(np.float32)
b = np.random.randn(bs).astype(np.float32)

scale_a, data_a = ref_encode_block(a)
scale_b, data_b = ref_encode_block(b)
golden = a + b

print("scale_a:", scale_a, " scale_b:", scale_b)
print("float32 a (first 4):", a[:4])
print("float32 b (first 4):", b[:4])
print("expected a+b (first 4):", golden[:4])

scale_a: 129  scale_b: 128
float32 a (first 4): [1.7640524 0.4001572 0.978738  2.2408931]
float32 b (first 4): [-0.88778573 -1.9807965  -0.34791216  0.15634897]
expected a+b (first 4): [ 0.87626666 -1.5806392   0.6308259   2.397242  ]


## Step 2 — Customize the Allo kernel

`mxfp8_block_add` is defined in `allo/library/mxfp8.py` using the Allo DSL.
It decodes both blocks to float32, adds element-wise, then re-encodes the result.

Pass the block size via `instantiate=[bs]`.

In [10]:
s = allo.customize(mxfp8.mxfp8_block_add, instantiate=[bs])
print(s.module)

module {
  func.func @decode_e8m0(%arg0: i8) -> f32 attributes {itypes = "u", otypes = "_"} {
    %0 = arith.extui %arg0 : i8 to i32
    %alloc = memref.alloc() {name = "u"} : memref<i32>
    affine.store %0, %alloc[] {to = "u"} : memref<i32>
    %cst = arith.constant 0.000000e+00 : f32
    %cst_0 = arith.constant 0.000000e+00 : f32
    %alloc_1 = memref.alloc() {name = "result"} : memref<f32>
    affine.store %cst_0, %alloc_1[] {to = "result"} : memref<f32>
    %1 = affine.load %alloc[] {from = "u"} : memref<i32>
    %c0_i32 = arith.constant 0 : i32
    %c0_i32_2 = arith.constant 0 : i32
    %2 = arith.cmpi ne, %1, %c0_i32_2 : i32
    %3 = affine.load %alloc[] {from = "u"} : memref<i32>
    %c255_i32 = arith.constant 255 : i32
    %c255_i32_3 = arith.constant 255 : i32
    %4 = arith.cmpi ne, %3, %c255_i32_3 : i32
    %5 = arith.andi %2, %4 : i1
    scf.if %5 {
      %7 = affine.load %alloc[] {from = "u"} : memref<i32>
      %8 = arith.extsi %7 : i32 to i33
      %c127_i32 = arith.con

## Step 3 — LLVM CPU simulation

Build with the default `llvm` target and run the kernel.

> **Note**: For scalar `uint8` arguments, pass Python `int` values (e.g. `int(scale_a)`),
> not `np.uint8` scalars.

In [11]:
mod = s.build()

scale_out = np.zeros(1, dtype=np.uint8)
data_out = np.zeros(bs, dtype=np.uint8)

mod(int(scale_a), data_a, int(scale_b), data_b, scale_out, data_out)

result = ref_decode_block(int(scale_out[0]), data_out)
print("Allo result (first 4):", result[:4])
print("Golden   (first 4):", golden[:4])

np.testing.assert_allclose(result, golden, atol=0.5)
print("LLVM simulation passed.")

Allo result (first 4): [ 0.875  -1.625   0.6875  2.5   ]
Golden   (first 4): [ 0.87626666 -1.5806392   0.6308259   2.397242  ]
LLVM simulation passed.


## Step 4 — Generate Vivado HLS C++

The same schedule can target `vhls` to emit synthesizable C++.
The generated top function is `mxfp8_block_add`.

Notebook output is often **truncated** (~2000 chars). To see the full code:

1. Run the cell below — it saves the complete C++ to a file and shows a scrollable view.
2. Open the saved `.cpp` file in your editor (path printed after save).

In [12]:
hls_mod = s.build(target="vhls")
hls_code = hls_mod.hls_code
lines = hls_code.splitlines()

# Save full generated code (best way to inspect everything)
hls_path = Path("mxfp8_block_add.hls.cpp")
hls_path.write_text(hls_code)

print(f"Generated {len(lines)} lines of HLS C++")
print(f"Full code saved to: {hls_path.resolve()}")
print("Tip: open that file in the editor if the notebook view is truncated.\n")

# Scrollable syntax-highlighted view (full text, not just an excerpt)
display(Code(hls_code, language="cpp"))

Generated 327 lines of HLS C++
Full code saved to: /home/rbdus0715/allo/tutorials/mxfp8_block_add.hls.cpp
Tip: open that file in the editor if the notebook view is truncated.



//===------------------------------------------------------------*- C++ -*-===//
//
// Automatically generated file for High-level Synthesis (HLS).
//
//===----------------------------------------------------------------------===//
#include <algorithm>
#include <ap_axi_sdata.h>
#include <ap_fixed.h>
#include <ap_int.h>
#include <hls_math.h>
#include <hls_stream.h>
#include <hls_vector.h>
#include <math.h>
#include <stdint.h>
using namespace std;
void decode_e8m0(
  uint8_t v0,
  float *v1
) {	// L2
  int32_t v2 = v0;	// L3
  int32_t u;	// L4
  u = v2;	// L5
  float result;	// L8
  result = (float)0.000000;	// L9
  int32_t v5 = u;	// L10
  bool v6 = v5 != 0;	// L13
  bool v7 = v5 != 255;	// L17
  bool v8 = v6 & v7;	// L18
  if (v8) {	// L19
    int32_t v9 = u;	// L20
    ap_int<33> v10 = v9;	// L21
    ap_int<33> v11 = v10 - 127;	// L25
    int32_t v12 = v11;	// L26
    int32_t exp;	// L27
    exp = v12;	// L28
    int32_t v14 = exp;	// L29
    float v15 = v14;	// L30
    float v16 = pow((float)2.000000, v15);	// L33
    result = v16;	// L34
  }
  *v1 = result;	// L36
}

void decode_e4m3(
  uint8_t v17,
  float *v18
) {	// L39
  int32_t v19 = v17;	// L40
  int32_t u1;	// L41
  u1 = v19;	// L42
  int32_t v21 = u1;	// L43
  int32_t v22 = v21 >> 7;	// L46
  int32_t v23 = v22 & 1;	// L49
  int32_t sign;	// L50
  sign = v23;	// L51
  int32_t v25 = u1;	// L52
  int32_t v26 = v25 >> 3;	// L55
  int32_t v27 = v26 & 15;	// L58
  int32_t exp1;	// L59
  exp1 = v27;	// L60
  int32_t v29 = u1;	// L61
  int32_t v30 = v29 & 7;	// L64
  int32_t mant;	// L65
  mant = v30;	// L66
  float val;	// L69
  val = (float)0.000000;	// L70
  int32_t v33 = exp1;	// L71
  bool v34 = v33 == 15;	// L74
  int32_t v35 = mant;	// L75
  bool v36 = v35 == 7;	// L78
  bool v37 = v34 & v36;	// L79
  if (v37) {	// L80
    val = (float)0.000000;	// L83
  } else {
    int32_t v38 = exp1;	// L85
    bool v39 = v38 == 0;	// L88
    if (v39) {	// L89
      int32_t v40 = mant;	// L90
      float v41 = v40;	// L91
      float v42 = v41 / (float)8.000000;	// L94
      float v43 = v42 * (float)0.015625;	// L106
      val = v43;	// L107
    } else {
      int32_t v44 = mant;	// L109
      float v45 = v44;	// L110
      float v46 = v45 / (float)8.000000;	// L113
      float v47 = v46 + (float)1.000000;	// L116
      int32_t v48 = exp1;	// L117
      ap_int<33> v49 = v48;	// L118
      ap_int<33> v50 = v49 - 7;	// L122
      float v51 = v50;	// L123
      float v52 = pow((float)2.000000, v51);	// L126
      float v53 = v47 * v52;	// L127
      val = v53;	// L128
    }
  }
  int32_t v54 = sign;	// L131
  bool v55 = v54 == 1;	// L134
  if (v55) {	// L135
    float v56 = val;	// L136
    float v57 = (float)0.000000 - v56;	// L139
    val = v57;	// L140
  }
  *v18 = val;	// L142
}

void encode_e8m0(
  float v58,
  uint8_t *v59
) {	// L145
  int32_t result1;	// L148
  result1 = 0;	// L149
  bool v61 = v58 > (float)0.000000;	// L152
  if (v61) {	// L153
    result1 = 254;	// L156
    l_S_e_0_e: for (int e = 0; e < 254; e++) {	// L157
      int v63 = (e + 1);	// L157
      int32_t v64 = result1;	// L158
      bool v65 = v64 == 254;	// L161
      if (v65) {	// L162
        ap_int<34> v66 = v63;	// L163
        ap_int<34> v67 = v66 - 127;	// L167
        float v68 = v67;	// L168
        float v69 = pow((float)2.000000, v68);	// L171
        float p;	// L172
        p = v69;	// L173
        float v71 = p;	// L174
        bool v72 = v71 >= v58;	// L175
        if (v72) {	// L176
          int32_t v73 = v63;	// L177
          result1 = v73;	// L178
        }
      }
    }
  }
  int32_t v74 = result1;	// L183
  *v59 = v74;	// L184
}

void encode_e4m3(
  float v75,
  uint8_t *v76
) {	// L187
  int32_t packed;	// L190
  packed = 0;	// L191
  bool v78 = v75 != (float)0.000000;	// L194
  if (v78) {	// L195
    int32_t sign1;	// L198
    sign1 = 0;	// L199
    float abs_f;	// L200
    abs_f = v75;	// L201
    bool v81 = v75 < (float)0.000000;	// L204
    if (v81) {	// L205
      sign1 = 1;	// L2

## Step 5 — Apply schedule and rebuild (optional)

Pipeline the decode/add and encode loops for better HLS throughput.
`schedule_mxfp8_block_add` is registered in `allo.library.KERNEL2SCHEDULE`.

In [13]:
s = mxfp8.schedule_mxfp8_block_add(s)
hls_sched = s.build(target="vhls")
hls_sched_code = hls_sched.hls_code

hls_sched_path = Path("mxfp8_block_add_scheduled.hls.cpp")
hls_sched_path.write_text(hls_sched_code)

assert "#pragma HLS pipeline" in hls_sched_code
print(f"Pipelined HLS: {len(hls_sched_code.splitlines())} lines")
print(f"Full code saved to: {hls_sched_path.resolve()}\n")

display(Code(hls_sched_code, language="cpp"))

Pipelined HLS: 330 lines
Full code saved to: /home/rbdus0715/allo/tutorials/mxfp8_block_add_scheduled.hls.cpp



//===------------------------------------------------------------*- C++ -*-===//
//
// Automatically generated file for High-level Synthesis (HLS).
//
//===----------------------------------------------------------------------===//
#include <algorithm>
#include <ap_axi_sdata.h>
#include <ap_fixed.h>
#include <ap_int.h>
#include <hls_math.h>
#include <hls_stream.h>
#include <hls_vector.h>
#include <math.h>
#include <stdint.h>
using namespace std;
void decode_e8m0(
  uint8_t v0,
  float *v1
) {	// L3
  int32_t v2 = v0;	// L4
  int32_t u;	// L5
  u = v2;	// L6
  float result;	// L8
  result = (float)0.000000;	// L9
  int32_t v5 = u;	// L10
  bool v6 = v5 != 0;	// L12
  bool v7 = v5 != 255;	// L14
  bool v8 = v6 & v7;	// L15
  if (v8) {	// L16
    int32_t v9 = u;	// L17
    ap_int<33> v10 = v9;	// L18
    ap_int<33> v11 = v10 - 127;	// L21
    int32_t v12 = v11;	// L22
    int32_t exp;	// L23
    exp = v12;	// L24
    int32_t v14 = exp;	// L25
    float v15 = v14;	// L26
    float v16 = pow((float)2.000000, v15);	// L28
    result = v16;	// L29
  }
  *v1 = result;	// L31
}

void decode_e4m3(
  uint8_t v17,
  float *v18
) {	// L34
  int32_t v19 = v17;	// L35
  int32_t u1;	// L36
  u1 = v19;	// L37
  int32_t v21 = u1;	// L38
  int32_t v22 = v21 >> 7;	// L40
  int32_t v23 = v22 & 1;	// L42
  int32_t sign;	// L43
  sign = v23;	// L44
  int32_t v25 = u1;	// L45
  int32_t v26 = v25 >> 3;	// L47
  int32_t v27 = v26 & 15;	// L49
  int32_t exp1;	// L50
  exp1 = v27;	// L51
  int32_t v29 = u1;	// L52
  int32_t v30 = v29 & 7;	// L53
  int32_t mant;	// L54
  mant = v30;	// L55
  float val;	// L57
  val = (float)0.000000;	// L58
  int32_t v33 = exp1;	// L59
  bool v34 = v33 == 15;	// L60
  int32_t v35 = mant;	// L61
  bool v36 = v35 == 7;	// L62
  bool v37 = v34 & v36;	// L63
  if (v37) {	// L64
    val = (float)0.000000;	// L65
  } else {
    int32_t v38 = exp1;	// L67
    bool v39 = v38 == 0;	// L69
    if (v39) {	// L70
      int32_t v40 = mant;	// L71
      float v41 = v40;	// L72
      float v42 = v41 / (float)8.000000;	// L74
      float v43 = v42 * (float)0.015625;	// L81
      val = v43;	// L82
    } else {
      int32_t v44 = mant;	// L84
      float v45 = v44;	// L85
      float v46 = v45 / (float)8.000000;	// L87
      float v47 = v46 + (float)1.000000;	// L89
      int32_t v48 = exp1;	// L90
      ap_int<33> v49 = v48;	// L91
      ap_int<33> v50 = v49 - 7;	// L93
      float v51 = v50;	// L94
      float v52 = pow((float)2.000000, v51);	// L96
      float v53 = v47 * v52;	// L97
      val = v53;	// L98
    }
  }
  int32_t v54 = sign;	// L101
  bool v55 = v54 == 1;	// L102
  if (v55) {	// L103
    float v56 = val;	// L104
    float v57 = (float)0.000000 - v56;	// L105
    val = v57;	// L106
  }
  *v18 = val;	// L108
}

void encode_e8m0(
  float v58,
  uint8_t *v59
) {	// L111
  int32_t result1;	// L113
  result1 = 0;	// L114
  bool v61 = v58 > (float)0.000000;	// L116
  if (v61) {	// L117
    result1 = 254;	// L119
    l_S_e_0_e: for (int e = 0; e < 254; e++) {	// L120
      int v63 = (e + 1);	// L121
      int32_t v64 = result1;	// L122
      bool v65 = v64 == 254;	// L123
      if (v65) {	// L124
        ap_int<34> v66 = v63;	// L125
        ap_int<34> v67 = v66 - 127;	// L128
        float v68 = v67;	// L129
        float v69 = pow((float)2.000000, v68);	// L131
        float p;	// L132
        p = v69;	// L133
        float v71 = p;	// L134
        bool v72 = v71 >= v58;	// L135
        if (v72) {	// L136
          int32_t v73 = v63;	// L137
          result1 = v73;	// L138
        }
      }
    }
  }
  int32_t v74 = result1;	// L143
  *v59 = v74;	// L144
}

void encode_e4m3(
  float v75,
  uint8_t *v76
) {	// L147
  int32_t packed;	// L149
  packed = 0;	// L150
  bool v78 = v75 != (float)0.000000;	// L152
  if (v78) {	// L153
    int32_t sign1;	// L154
    sign1 = 0;	// L155
    float abs_f;	// L156
    abs_f = v75;	// L157
    bool v81 = v75 < (float)0.000000;	// L158
    if (v81) {	// L159
      sign1 = 1;	// L161
      floa

## Step 6 — Vitis HLS (optional, requires toolchain)

If Vitis HLS is installed and configured, you can also run:

```python
mod = s.build(target="vitis_hls", mode="csim", project="mxfp8.prj")
mod(int(scale_a), data_a, int(scale_b), data_b, scale_out, data_out)
```

Uncomment and run the cell below only when the Vitis environment is available.

In [ ]:
# import allo.backend.hls as hls
#
# if hls.is_available("vitis_hls"):
#     mod = s.build(target="vitis_hls", mode="csim", project="mxfp8.prj")
#     scale_out = np.zeros(1, dtype=np.uint8)
#     data_out = np.zeros(bs, dtype=np.uint8)
#     mod(int(scale_a), data_a, int(scale_b), data_b, scale_out, data_out)
#     result = ref_decode_block(int(scale_out[0]), data_out)
#     np.testing.assert_allclose(result, golden, atol=0.5)
#     print("Vitis HLS csim passed.")
# else:
#     print("Vitis HLS not available — skip this step.")

## Summary

| Component | Role |
|-----------|------|
| `ref_*` functions | Python/NumPy golden model for verification |
| `decode_e4m3`, `encode_e4m3`, … | Allo DSL scalar helpers (compiled into HLS sub-functions) |
| `mxfp8_block_add` | Top-level kernel: decode → add → encode |
| `schedule_mxfp8_block_add` | Pipeline loops for HLS |

**Allo DSL tips for custom numeric formats:**

- Use existing types (`uint8`, `float32`) — no new MLIR type required
- Avoid mid-function `return` statements (use a single return at the end)
- Library kernels in `allo/library/` compile through the same LLVM / HLS flow as `gemv` or `nn.linear2d`